## Class Assignment 4

# Fine-tuning a LoRA Adapter on GPT-2 for a Q&A Dataset

Fine-tune a **LoRA adapter** on top of a **GPT-2 model** for a custom Question-Answering dataset.

Steps:

1. Choose a small Q&A dataset suitable for Google Colab T4 GPU.
2. Convert the dataset into a text-to-text causal language modeling format.
3. Fine-tune GPT-2 using LoRA.
4. Evaluate the trained model using **perplexity**.
5. Generate sample Q&A outputs.
6. Discuss GPU memory usage and training efficiency.

---

## Chosen Dataset

I have used **SQuAD** dataset from Hugging Face.

SQuAD is a well-known Question-Answering dataset containing questions, context passages, and answers.

For this assignment, I have **not** trained the model on the full dataset because Google Colab T4 has limited memory and runtime.  
Instead, I have used a small subset:

- Training samples: 2,000
- Validation samples: 300

This keeps training efficient while still demonstrating LoRA-based fine-tuning properly.

---

SQuAD is suitable because:

- It is a real Q&A dataset.
- Each example has a question, context, and answer.
- It can be converted easily into a prompt-completion format.
- A small subset can be trained efficiently on a Colab T4 GPU.
- LoRA is parameter-efficient, so only a small number of additional adapter weights are trained.


In [19]:
# ============================================================
# Step 1: Install required libraries
# ============================================================

# transformers  -> provides GPT-2 model and tokenizer
# datasets      -> helps us load the SQuAD dataset
# peft          -> provides LoRA implementation
# accelerate    -> helps with efficient training on GPU
# bitsandbytes  -> optional library often useful for memory-efficient training

!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q -U transformers datasets peft accelerate bitsandbytes "torchao>0.16.0"

In [20]:
# ============================================================
# Step 2: Import required libraries
# ============================================================

import math
import torch
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device being used:", device)

if device == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory available in GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("GPU not available. Please enable GPU from Runtime > Change runtime type > GPU")

Device being used: cuda
GPU name: Tesla T4
GPU memory available in GB: 14.56


## Dataset Loading and Preparation

GPT-2 is a causal language model. It does not naturally take separate question and answer fields like a normal Q&A model.

So we convert every example into a single text format like this:

```text
Context: ...
Question: ...
Answer: ...
```

During training, GPT-2 learns to predict the next token in this sequence.

Later, when we provide a context and question, the model should continue the text by generating an answer.


In [21]:
# ============================================================
# Step 3: Load a small subset of the SQuAD dataset
# ============================================================

# Only a small subset is used to keep training practical on Colab T4.
# Full SQuAD is much larger, but for this assignment a smaller subset is enough.

raw_dataset = load_dataset("squad")

train_data = raw_dataset["train"].select(range(2000))
valid_data = raw_dataset["validation"].select(range(300))

print(train_data)
print(valid_data)

# Show one raw example
train_data[0]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 2000
})
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})


{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

In [22]:
# ============================================================
# Step 4: Convert SQuAD examples into GPT-2 training text
# ============================================================

def format_qa_example(example):
    """
    Converts one SQuAD example into a single text prompt.

    GPT-2 is trained as a causal language model, so we give it
    context, question, and answer in one continuous text sequence.
    """

    context = example["context"]
    question = example["question"]

    # SQuAD stores answers as a dictionary.
    # We take the first answer text.
    answer = example["answers"]["text"][0]

    formatted_text = (
        "Context: " + context + "\n"
        "Question: " + question + "\n"
        "Answer: " + answer
    )

    return {"text": formatted_text}


train_text = train_data.map(format_qa_example)
valid_text = valid_data.map(format_qa_example)

# Display a formatted example
print(train_text[0]["text"])

Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer: Saint Bernadette Soubirous


## Tokenization

GPT-2 works with tokens, not raw text.

Important detail: GPT-2 does not have a default padding token.  
So we set its padding token equal to the end-of-sequence token.


In [23]:
# ============================================================
# Step 5: Load GPT-2 tokenizer
# ============================================================

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 does not have a default pad token.
# We use the EOS token as the pad token.
tokenizer.pad_token = tokenizer.eos_token

# Maximum sequence length.
# 256 is a practical choice for Colab T4.
# Longer sequences use more memory.
max_length = 256


def tokenize_function(example):
    """
    Tokenizes the formatted text.
    truncation=True means very long examples are shortened.
    padding='max_length' makes all examples the same length.
    """

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )


tokenized_train = train_text.map(tokenize_function, batched=True, remove_columns=train_text.column_names)
tokenized_valid = valid_text.map(tokenize_function, batched=True, remove_columns=valid_text.column_names)

print(tokenized_train)
print(tokenized_valid)

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2000
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 300
})


## Load GPT-2 and Apply LoRA

LoRA stands for **Low-Rank Adaptation**.

Instead of updating all GPT-2 parameters, LoRA adds small trainable matrices to selected layers.  
This makes training:

- faster,
- cheaper,
- more memory-efficient,
- suitable for limited GPUs like Colab T4.

For GPT-2, common target modules are:

- `c_attn`
- `c_proj`

These are attention-related projection layers inside GPT-2.


In [24]:
# ============================================================
# Step 6: Load GPT-2 model
# ============================================================

model = AutoModelForCausalLM.from_pretrained(model_name)

# Very important:
# GPT-2 needs to know the padding token id.
model.config.pad_token_id = tokenizer.pad_token_id

# Move model to GPU if available
model.to(device)

print("GPT-2 model loaded successfully.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 model loaded successfully.


In [25]:
# ============================================================
# Step 7: Configure LoRA
# ============================================================

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,

    # r is the rank of the LoRA matrices.
    # Smaller r = fewer trainable parameters.
    # 8 is a good lightweight value for Colab.
    r=8,

    # alpha controls the scaling of LoRA updates.
    lora_alpha=16,

    # Dropout helps reduce overfitting.
    lora_dropout=0.05,

    # GPT-2 attention projection layers where LoRA is applied.
    target_modules=["c_attn", "c_proj"],

    # We are only training LoRA adapter weights.
    bias="none"
)

# Wrap the GPT-2 model with LoRA adapters
model = get_peft_model(model, lora_config)

# Show number of trainable parameters
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


## LoRA Configuration and Hyperparameters

The configuration used in this notebook is:

| Setting | Value |
|---|---|
| Base model | GPT-2 |
| Dataset | SQuAD subset |
| Training samples | 2,000 |
| Validation samples | 300 |
| LoRA rank `r` | 8 |
| LoRA alpha | 16 |
| LoRA dropout | 0.05 |
| Target modules | `c_attn`, `c_proj` |
| Epochs | 1 |
| Batch size | 2 |
| Gradient accumulation | 8 |
| Effective batch size | 16 |
| Learning rate | 2e-4 |
| Max sequence length | 256 |

These settings are intentionally small so the notebook can run on a free/standard Google Colab T4 GPU.


In [26]:
# ============================================================
# Step 8: Create data collator
# ============================================================

# For causal language modeling, labels are usually the same as input_ids.
# This collator prepares batches for GPT-2 training.

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # GPT-2 uses causal language modeling, not masked language modeling
)

In [27]:
# ============================================================
# Step 9: Define training arguments
# ============================================================

training_args = TrainingArguments(
    output_dir="./gpt2-squad-lora",

    # One epoch is enough for assignment demonstration.
    # You may increase this if you have more GPU time.
    num_train_epochs=1,

    # Small batch size to avoid GPU memory issues on T4.
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    # Accumulate gradients to simulate a larger batch size.
    gradient_accumulation_steps=8,

    # LoRA often works well with a slightly higher learning rate than full fine-tuning.
    learning_rate=2e-4,

    # Evaluate during training.
    eval_strategy="steps",
    eval_steps=100,

    # Save checkpoints occasionally.
    save_steps=100,
    save_total_limit=2,

    # Logging frequency.
    logging_steps=25,

    # Mixed precision reduces GPU memory usage and speeds up training on T4.
    fp16=torch.cuda.is_available(),

    # Useful for Colab.
    report_to="none",

    # Keep training simple and stable.
    warmup_steps=20,
    weight_decay=0.01
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=100,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False

In [28]:
# ============================================================
# Step 10: Create Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator
)

print("Trainer is ready.")

Trainer is ready.


In [29]:
# ============================================================
# Step 11: Train the LoRA adapter
# ============================================================

# This cell may take some time on Colab T4.
# Expected runtime depends on GPU availability and Colab load.

train_result = trainer.train()

print("Training completed.")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,3.247136,2.418996


Training completed.


In [30]:
# ============================================================
# Step 12: Evaluate the trained model and calculate perplexity
# ============================================================

eval_result = trainer.evaluate()

eval_loss = eval_result["eval_loss"]
perplexity = math.exp(eval_loss)

print("Evaluation loss:", eval_loss)
print("Final perplexity:", perplexity)

eval_result

Evaluation loss: 2.417433500289917
Final perplexity: 11.217033826337163


{'eval_loss': 2.417433500289917,
 'eval_runtime': 4.3906,
 'eval_samples_per_second': 68.328,
 'eval_steps_per_second': 34.164,
 'epoch': 1.0}

## Interpreting Perplexity

Perplexity measures how well the model predicts the next token.

Lower perplexity means the model is more confident and better at predicting the validation text.

Because we train only on a small subset and for one epoch, the perplexity may not be extremely low.  
The goal here is to demonstrate efficient LoRA fine-tuning in a Colab-friendly setup.


In [31]:
# ============================================================
# Step 13: Generate sample Q&A outputs
# ============================================================

def generate_answer(context, question, max_new_tokens=60):
    """
    Generates an answer using the fine-tuned GPT-2 LoRA model.

    We give the model context and question.
    Then the model continues after 'Answer:'.
    """

    prompt = (
        "Context: " + context + "\n"
        "Question: " + question + "\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return generated_text


# Sample 1
context_1 = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars "
    "in Paris, France. It is named after the engineer Gustave Eiffel."
)

question_1 = "Where is the Eiffel Tower located?"

print(generate_answer(context_1, question_1))

Context: The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel.
Question: Where is the Eiffel Tower located?
Answer: in the city of Paris.
Question: Where is the Eiffel Tower located? Answer: in the city of Paris.
Question: What is the meaning of the word "Eiffel"?
Answer: it means "the building of a house".
Question: Where is the E


In [32]:
# Sample 2

context_2 = (
    "Python is a high-level programming language. It is widely used in "
    "data science, web development, automation, and machine learning."
)

question_2 = "What is Python used for?"

print(generate_answer(context_2, question_2))

Context: Python is a high-level programming language. It is widely used in data science, web development, automation, and machine learning.
Question: What is Python used for?
Answer: In the past, the term "Python" was used to describe a programming language used by the ancient Greeks. In the modern language, "Python" has become a popular term for many languages, especially in the United States and Europe.
Question: What is Python?
Answer: Python is a


In [33]:
# Sample 3

context_3 = (
    "The Amazon rainforest is the largest tropical rainforest in the world. "
    "It is located in South America and is known for its rich biodiversity."
)

question_3 = "What is the Amazon rainforest known for?"

print(generate_answer(context_3, question_3))

Context: The Amazon rainforest is the largest tropical rainforest in the world. It is located in South America and is known for its rich biodiversity.
Question: What is the Amazon rainforest known for?
Answer: It contains over half a million species of flora and fauna, including the Amazon rainforest's most valuable rainforest, the Amazonian rainforest.
Question: What is the Amazon rainforest known for?
Answer: The Amazon rainforest is known for its rich biodiversity and for its abundance of


## GPU Memory Usage and Training Efficiency Discussion

This experiment is designed to fit within Google Colab T4 GPU limits.

### Why LoRA is efficient

Full fine-tuning updates all GPT-2 parameters.  
That requires more GPU memory and more training time.

LoRA updates only a small number of adapter parameters.  
The original GPT-2 weights remain mostly frozen.

This gives several benefits:

- Lower GPU memory usage
- Faster training
- Smaller saved adapter files
- Easier experimentation on Colab

### Why this setup works on T4

A Colab T4 GPU usually has around 15 GB of GPU memory.  
This notebook keeps memory usage controlled by using:

- GPT-2 small model
- LoRA instead of full fine-tuning
- Batch size of 2
- Gradient accumulation
- FP16 mixed precision
- Maximum sequence length of 256
- Small dataset subset

### Expected training behavior

The training should run efficiently on Colab T4.  
The final perplexity depends on random seed, GPU runtime, dataset subset, and number of epochs.

For better results, students can try:

- increasing training samples,
- training for 2–3 epochs,
- increasing max sequence length,
- tuning LoRA rank,
- using a cleaner instruction-style Q&A dataset.

But these changes may increase training time and GPU memory usage.


In [34]:
# ============================================================
# Step 14: Optional - Check GPU memory usage
# ============================================================

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3

    print("GPU memory allocated in GB:", round(allocated, 2))
    print("GPU memory reserved in GB:", round(reserved, 2))
else:
    print("GPU memory check skipped because CUDA is not available.")

GPU memory allocated in GB: 0.5
GPU memory reserved in GB: 1.54


In [35]:
# ============================================================
# Step 15: Save the LoRA adapter and tokenizer
# ============================================================

# This saves only the LoRA adapter weights, not the full GPT-2 model.
# That is one of the benefits of parameter-efficient fine-tuning.

model.save_pretrained("./gpt2-squad-lora-adapter")
tokenizer.save_pretrained("./gpt2-squad-lora-adapter")

print("LoRA adapter and tokenizer saved successfully.")

LoRA adapter and tokenizer saved successfully.


# Final Summary

## Dataset

The selected dataset is SQuAD. It is a standard Q&A dataset containing context passages, questions, and answers.  
A subset of 2,000 training examples and 300 validation examples was used to fit within Google Colab T4 GPU limitations.

## Model

The base model is GPT-2. Since GPT-2 is a causal language model, the Q&A examples were converted into prompt-completion text format.

## LoRA Settings

LoRA was applied to GPT-2 attention projection layers using:

- Rank: 8
- Alpha: 16
- Dropout: 0.05
- Target modules: `c_attn`, `c_proj`

## Training Settings

The model was trained for 1 epoch using:

- Learning rate: 2e-4
- Batch size: 2
- Gradient accumulation steps: 8
- FP16 mixed precision
- Max sequence length: 256

## Perplexity

The final perplexity is calculated as:

```python
perplexity = math.exp(eval_loss)
```

Run the evaluation cell after training and report the printed final perplexity value.

## Efficiency

The experiment is efficient because LoRA trains only a small number of adapter parameters while keeping most GPT-2 weights frozen.  
This makes the approach suitable for Google Colab T4 GPU.
